TITLE

Here, we select 

Why SentenceTransformer?
Because it gives you sentence-level embeddings out of the box. You’re working with short titles, not long documents, so a model built to represent whole sentences/posts is a better fit than older word embeddings like Word2Vec/GloVe, and much simpler than building your own transformer workflow with raw Hugging Face models.

Are there other options?
Yes:

TF-IDF: simpler, cheaper, but only word/phrase matching
Word2Vec / GloVe / FastText: older embedding approaches, usually more fiddly and less strong than modern sentence embeddings
Raw transformer models from Hugging Face: more flexible, but more setup
OpenAI / API embeddings: good option too, but adds API dependency/cost

We pick all-MiniLM-L6-v2 because it’s a strong practical baseline: good quality, fast to run locally, and lightweight enough for experiments on your laptop. That’s what I meant by “small, fast, good baseline” — not “best possible”, but “sensible first choice”.

An embeddings baseline is created with just title in the feature set for comparison purposes (subreddit context will be added later if successful).



In [10]:
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

In [11]:
# Load shortlist_combined_df from previous notebook
shortlist_combined_df = pd.read_parquet("../data/processed/shortlist_combined_df.parquet")

print("✅ Data loaded successfully. Sample data:")
display(shortlist_combined_df.head())

✅ Data loaded successfully. Sample data:


,id,title,score,upvote_ratio,num_comments,created_utc,subreddit,subscribers,permalink,url,...,filename,score_scaled,comments_scaled,engagement,high_engagement,engagement_rank,title_length,has_question,has_exclamation,has_number
0,ablzuq,"People who haven't pooped in 2019 yet, why are...",221996,0.91,7938,1/2/2019 2:36,askreddit,48248325,https://www.reddit.com/r/AskReddit/comments/ab...,https://www.reddit.com/r/AskReddit/comments/ab...,...,../data/askreddit.csv,3.774641,0.515065,4.289706,1,0.985150,87,1,0,1
1,l7530r,How would you feel about Reddit adding 3 NSFW ...,217921,0.87,2894,1/29/2021 0:21,askreddit,48248325,https://www.reddit.com/r/AskReddit/comments/l7...,https://www.reddit.com/r/AskReddit/comments/l7...,...,../data/askreddit.csv,3.675889,-0.170698,3.505191,1,0.975202,208,1,0,1
2,f08dxb,Would you watch a show where a billionaire CEO...,197595,0.90,13342,2/7/2020 15:23,askreddit,48248325,https://www.reddit.com/r/AskReddit/comments/f0...,https://www.reddit.com/r/AskReddit/comments/f0...,...,../data/askreddit.csv,3.183320,1.249772,4.433092,1,0.986448,208,1,0,0
3,iwedc5,"What if God came down one day and said ""It's p...",195927,0.92,10250,9/20/2020 19:31,askreddit,48248325,https://www.reddit.com/r/AskReddit/comments/iw...,https://www.reddit.com/r/AskReddit/comments/iw...,...,../data/askreddit.csv,3.142898,0.829396,3.972294,1,0.982555,72,1,0,0
4,draola,How would you feel about a feature where if so...,186441,0.90,2777,11/4/2019 7:27,askreddit,48248325,https://www.reddit.com/r/AskReddit/comments/dr...,https://www.reddit.com/r/AskReddit/comments/dr...,...,../data/askreddit.csv,2.913019,-0.186605,2.726414,1,0.958333,116,1,0,0


In [12]:
# Load embedding model
embed = SentenceTransformer('all-MiniLM-L6-v2')

# Test on some sample sentences
test_embeddings = embed.encode(["hello world", "my dog is cute"])
test_embeddings.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(2, 384)

In [13]:
# Small test on actual data
sample_titles = shortlist_combined_df["title"].head(100).tolist()
sample_embeddings = embed.encode(sample_titles)

print("✅ Sample embeddings generated. Shape:", sample_embeddings.shape)
print("✅ Embeddings for first 5 titles:")
for i in range(5):
    print(f"Title: {sample_titles[i]}")
    print(f"Embedding (first 5 values): {sample_embeddings[i][:5]}")
    print()

✅ Sample embeddings generated. Shape: (100, 384)
✅ Embeddings for first 5 titles:
Title: People who haven't pooped in 2019 yet, why are you still holding on to last years shit?
Embedding (first 5 values): [ 0.03596077 -0.04791442  0.12893978  0.03466046  0.05104578]

Title: How would you feel about Reddit adding 3 NSFW filters to distinguish between porn, gore, and repetitive posts asking how you would feel about Reddit adding 2 NSFW filters to distinguish between porn and gore?
Embedding (first 5 values): [ 0.01805317 -0.07294483 -0.06138439 -0.02201001  0.10958029]

Title: Would you watch a show where a billionaire CEO has to go an entire month on their lowest paid employees salary, without access to any other resources than that of the employee? What do you think would happen?
Embedding (first 5 values): [-0.01524418  0.02096531 -0.01706846 -0.02929186 -0.00502668]

Title: What if God came down one day and said "It's pronounced 'Jod' then left?
Embedding (first 5 values): [ 0.022951

In [ ]:
# Encode titles in batches
titles = shortlist_combined_df["title"].tolist()

X_embeddings = embed.encode(
    titles,
    batch_size=32,
    show_progress_bar=True
)

print(X_embeddings.shape)

Batches:   0%|          | 0/217 [00:00<?, ?it/s]

(6936, 384)


In [15]:
# Define target variable
y = shortlist_combined_df["high_engagement"]

# Split data into train/test sets
from sklearn.model_selection import train_test_split

X_embeddings_train, X_embeddings_test, y_train, y_test = train_test_split(
    X_embeddings,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Train logistic regression model with class weights
embedding_model= LogisticRegression(max_iter=1000, class_weight="balanced")
embedding_model.fit(X_embeddings_train, y_train)

y_embeddings_pred = embedding_model.predict(X_embeddings_test)

print("Embedding Model")
print(classification_report(y_test, y_embeddings_pred))
print(confusion_matrix(y_test, y_embeddings_pred))

Embedding Model
              precision    recall  f1-score   support

           0       0.89      0.75      0.81      1041
           1       0.49      0.71      0.58       347

    accuracy                           0.74      1388
   macro avg       0.69      0.73      0.70      1388
weighted avg       0.79      0.74      0.75      1388

[[779 262]
 [ 99 248]]


#### 💡 Embeddings baseline

As mentioned above, to fairly assess the impact of text representation, embeddings are evaluated using title-only input, consistent with the earlier TF-IDF baseline. 

Under this setup, embeddings substantially improve recall (0.38 → 0.71), indicating that they capture meaningful semantic signal beyond simple word matching.

## Embeddings + Context

In [19]:
# --- Define new feature set with embeddings and subreddit context ---

# Create column names for each embedding dimension
embedding_dim = X_embeddings.shape[1]
embedding_columns = [f"embedding_{i}" for i in range(embedding_dim)]

# Create a new DataFrame with embedding columns
embeddings_df = pd.DataFrame(X_embeddings, columns=embedding_columns, index=shortlist_combined_df.index)

# Create new feature set by combining embeddings with subreddit context
X_embed_context = pd.concat([embeddings_df, shortlist_combined_df[["subreddit"]]], axis=1)

X_embed_context.shape

(6936, 385)

In [ ]:
# Split
X_embed_context_train, X_embed_context_test, y_train, y_test = train_test_split(
    X_embed_context,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Preprocess subreddit context using one-hot encoding
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

embed_context_preprocessor = ColumnTransformer(
    transformers=[
        ("subreddit_ohe", OneHotEncoder(handle_unknown="ignore"), ["subreddit"]),
        ("embedding_pass", "passthrough", embedding_columns)
    ]
)

# Create pipeline for logistic regression with embedding + context features
from sklearn.pipeline import Pipeline

embedding_context_pipeline = Pipeline(steps=[
    ("preprocessor", embed_context_preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

# Train model
embedding_context_pipeline.fit(X_embed_context_train, y_train)

# Evaluate model
y_embed_context_pred = embedding_context_pipeline.predict(X_embed_context_test)

print("Embedding + Context Model")
print(classification_report(y_test, y_embed_context_pred))
print(confusion_matrix(y_test, y_embed_context_pred))

Embedding + Context Model
              precision    recall  f1-score   support

           0       0.95      0.78      0.85      1041
           1       0.56      0.87      0.68       347

    accuracy                           0.80      1388
   macro avg       0.76      0.82      0.77      1388
weighted avg       0.85      0.80      0.81      1388

[[808 233]
 [ 45 302]]


Combining embeddings with subreddit context leads to the highest recall for high-engagement posts (0.87), further reducing the number of missed high-performing posts.

However, this comes at the cost of precision, with a higher number of low-engagement posts incorrectly classified as high. This reflects the model’s increased tendency to prioritise recall, continuing the trade-off introduced through class weighting.

Overall, the results suggest that embeddings add meaningful semantic signal on top of contextual features, but that improving recall in this way requires accepting a higher rate of false positives. This trade-off may be appropriate in scenarios where missing high-performing content is more costly than promoting weaker posts.

In [22]:
# Save best model for future use
import joblib
joblib.dump(embedding_context_pipeline, "../models/embedding_context_engagement_model.joblib")

['../models/embedding_context_engagement_model.joblib']